# ChemistryDatabase: create -> extend -> import

Demonstrates the user-facing `ChemistryDatabase` lifecycle:

1. **Import a stock database** -- `AD_BASIC` provides CO2/NH4+ aqueous
   chemistry plus cross-phase CO2 partition out of the box.
2. **Extend with custom species** -- add a model-specific VFA (butyric acid)
   without touching the stock module.
3. **Override the `ThermoFramework`** -- switch to activity-corrected Davies
   model for a higher-accuracy run.
4. **Inspect the result** -- iterate reactions and species to confirm the
   composition.

The demo is self-contained and produces no simulation output; it exercises
the chemistry declaration API only.

## 1. Import a stock database

In [1]:
from PyOMES.databases.anaerobic_digestion import AD_BASIC
from PyOMES.databases.aqueous import AQUEOUS_DEFAULT

print("=== 1. Stock databases ===")
print(f"AQUEOUS_DEFAULT species : {sorted(AQUEOUS_DEFAULT.species.keys())}")
aq_labels = [r.label for r in AQUEOUS_DEFAULT.reactions]
print(f"AQUEOUS_DEFAULT reactions: {aq_labels}")
print()
ad_labels = [r.label for r in AD_BASIC.reactions]
print(f"AD_BASIC reactions       : {ad_labels}")

=== 1. Stock databases ===
AQUEOUS_DEFAULT species : ['CO2', 'CO3--', 'H+', 'H2O', 'HCO3-', 'NH3', 'NH4+', 'OH-']
AQUEOUS_DEFAULT reactions: ['eq_water', 'eq_CO2', 'eq_NH4']

AD_BASIC reactions       : ['eq_water', 'eq_CO2', 'eq_NH4', 'eq_phosphate_1', 'eq_phosphate_2', 'eq_phosphate_3', 'eq_bisulfate', 'partition_CO2', 'eq_H2S']


## 2. Extend with a custom species

`ChemistryDatabase.extend()` returns a new database with the added species
merged in -- the stock `AD_BASIC` is left untouched.

In [2]:
from PyOMES.chemistry import Species

ButyricAcid = Species(
    id="ButyricAcid",
    atoms={"C": 4, "H": 8, "O": 2},
    charge=0,
    MW=88.106,
)
Butyrate = Species(
    id="Butyrate-",
    atoms={"C": 4, "H": 7, "O": 2},
    charge=-1,
    MW=87.098,
)

MY_DB = AD_BASIC.extend(
    species={
        "ButyricAcid": ButyricAcid,
        "Butyrate-":   Butyrate,
    },
)

print("=== 2. Extended database ===")
print(f"MY_DB species count: {len(MY_DB.species)}")
assert "ButyricAcid" in MY_DB.species, "ButyricAcid missing from extended DB"
assert "CO2" in MY_DB.species,         "CO2 unexpectedly dropped by extension"
print(f"MY_DB has ButyricAcid: {ButyricAcid.id!r}")
print(f"MY_DB has CO2:         {MY_DB.species['CO2'].id!r}")

=== 2. Extended database ===
MY_DB species count: 21
MY_DB has ButyricAcid: 'ButyricAcid'
MY_DB has CO2:         'CO2'


## 3. Override the `ThermoFramework`

`ThermoFramework` is a frozen dataclass -- runtime overrides go through
`dataclasses.replace()`. The non-ideality model itself is set via
`liquid_activity=` (a `LiquidPhaseModel` instance, e.g. `DaviesLiquidModel()`);
`activity_model` is a derived read-only property (the model's name label),
not a settable kwarg.

In [3]:
import dataclasses
from PyOMES.thermo import ThermoFramework, DaviesLiquidModel

activity_thermo = dataclasses.replace(
    AD_BASIC.thermo,
    liquid_activity=DaviesLiquidModel(),
)
MY_DB_ACTIVE = MY_DB.extend(thermo=activity_thermo)

print("=== 3. Activity-corrected database ===")
print(f"AD_BASIC.thermo.activity_model    : {AD_BASIC.thermo.activity_model}")
print(f"MY_DB_ACTIVE.thermo.activity_model: {MY_DB_ACTIVE.thermo.activity_model}")
assert MY_DB_ACTIVE.thermo.activity_model == "davies"
assert AD_BASIC.thermo.activity_model == "ideal"  # original unchanged

=== 3. Activity-corrected database ===
AD_BASIC.thermo.activity_model    : ideal
MY_DB_ACTIVE.thermo.activity_model: davies


## 4. Inspect reaction composition

In [4]:
print("=== 4. Reaction composition of MY_DB_ACTIVE ===")
for rxn in MY_DB_ACTIVE.reactions:
    label = rxn.label or "(unlabelled)"
    phases = {e.phase for e in rxn.stoichiometry}
    sp_ids = [e.species.id for e in rxn.stoichiometry]
    print(f"  {label:20s}  phases={sorted(phases)}  species={sp_ids}")

=== 4. Reaction composition of MY_DB_ACTIVE ===
  eq_water              phases=['liquid']  species=['H2O', 'H+', 'OH-']
  eq_CO2                phases=['liquid']  species=['CO2', 'H2O', 'HCO3-', 'H+']
  eq_NH4                phases=['liquid']  species=['NH4+', 'NH3', 'H+']
  eq_phosphate_1        phases=['liquid']  species=['H3PO4', 'H2PO4-', 'H+']
  eq_phosphate_2        phases=['liquid']  species=['H2PO4-', 'HPO4--', 'H+']
  eq_phosphate_3        phases=['liquid']  species=['HPO4--', 'PO4---', 'H+']
  eq_bisulfate          phases=['liquid']  species=['HSO4-', 'SO4--', 'H+']
  partition_CO2         phases=['gas', 'liquid']  species=['CO2', 'CO2']
  eq_H2S                phases=['liquid']  species=['H2S', 'HS-', 'H+']


## 5. pKa temperature correction via `ThermoFramework`

`ThermoFramework.pKa_at_T()` applies a Van 't Hoff correction given a
reference pKa and a dissociation enthalpy.

In [5]:
print("=== 5. Temperature correction ===")
tf = MY_DB_ACTIVE.thermo
pKa_CO2_25 = 6.35
dH_CO2 = 7646.0  # J/mol (BSM2-canonical)
pKa_CO2_35 = tf.pKa_at_T(pKa_ref=pKa_CO2_25, dH_J_per_mol=dH_CO2, T_K=308.15)
print(f"CO2 pKa1 at 25 C: {pKa_CO2_25:.4f}")
print(f"CO2 pKa1 at 35 C: {pKa_CO2_35:.4f}  (Van 't Hoff correction)")
assert pKa_CO2_35 < pKa_CO2_25, "pKa should decrease with temperature for CO2"

print()
print("All assertions passed. ChemistryDatabase lifecycle demo complete.")

=== 5. Temperature correction ===
CO2 pKa1 at 25 C: 6.3500
CO2 pKa1 at 35 C: 6.3065  (Van 't Hoff correction)

All assertions passed. ChemistryDatabase lifecycle demo complete.
